# House Price Prediction using Machine Learning

**Dataset:** House Sales in King County, USA  
**Original source:** https://www.kaggle.com/datasets/shree1992/housedata

This notebook loads the accompanying `House_Prices_King_County.xlsx` file, cleans the data, performs EDA with six visualizations, engineers features, and compares Linear Regression, Decision Tree, and Random Forest models.

## 1. Import libraries

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

sns.set_theme(style='whitegrid', palette='deep')
pd.set_option('display.max_columns', None)

## 2. Load and inspect the dataset

In [ ]:
file_name = 'House_Prices_King_County.xlsx'
df = pd.read_excel(file_name, sheet_name='House Data')

print(f'Dataset shape: {df.shape}')
display(df.head())
df.info()
display(df.describe().T)

## 3. Data cleaning
The code removes duplicate rows, parses the sale date, and reports missing values. The preprocessing pipeline below also imputes any missing values safely before model training.

In [ ]:
print('Missing values before cleaning:')
display(df.isnull().sum()[df.isnull().sum() > 0])
print('Duplicate rows before cleaning:', df.duplicated().sum())

df = df.drop_duplicates().copy()
df['date'] = pd.to_datetime(df['date'], errors='coerce')
print('Rows after removing duplicates:', len(df))
print('Missing dates:', df['date'].isna().sum())

## 4. Exploratory data analysis (six visualizations)

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(20, 11))

# 1. Distribution of house prices
sns.histplot(df['price'], bins=50, kde=True, ax=axes[0, 0], color='steelblue')
axes[0, 0].set_title('1. Distribution of House Prices')

# 2. Living area versus price
sns.scatterplot(data=df.sample(min(5000, len(df)), random_state=42), x='sqft_living', y='price', alpha=0.45, ax=axes[0, 1])
axes[0, 1].set_title('2. Living Area vs Price')

# 3. Bedrooms versus price
sns.boxplot(data=df[df['bedrooms'] <= 8], x='bedrooms', y='price', ax=axes[0, 2])
axes[0, 2].set_title('3. Price by Number of Bedrooms')

# 4. Grade versus price
sns.boxplot(data=df, x='grade', y='price', ax=axes[1, 0])
axes[1, 0].set_title('4. Price by Construction Grade')

# 5. Correlation heatmap
corr_cols = ['price', 'sqft_living', 'sqft_lot', 'bathrooms', 'grade', 'yr_built', 'lat', 'long']
sns.heatmap(df[corr_cols].corr(), cmap='coolwarm', center=0, annot=True, fmt='.2f', ax=axes[1, 1])
axes[1, 1].set_title('5. Correlation Heatmap')

# 6. Waterfront versus price
sns.barplot(data=df, x='waterfront', y='price', estimator=np.mean, errorbar=None, ax=axes[1, 2])
axes[1, 2].set_title('6. Average Price: Waterfront vs Non-waterfront')
axes[1, 2].set_xlabel('Waterfront (0 = No, 1 = Yes)')

for ax in axes.flat:
    ax.ticklabel_format(style='plain', axis='y')
plt.tight_layout()
plt.show()

## 5. Feature engineering and encoding
Date parts, house age, renovation flag, and total usable area are engineered. `zipcode` is treated as a categorical feature and one-hot encoded.

In [ ]:
model_df = df.copy()
model_df['sale_year'] = model_df['date'].dt.year
model_df['sale_month'] = model_df['date'].dt.month
model_df['house_age'] = model_df['sale_year'] - model_df['yr_built']
model_df['was_renovated'] = (model_df['yr_renovated'] > 0).astype(int)
model_df['total_sqft'] = model_df['sqft_living'] + model_df['sqft_basement']

# id is an identifier rather than a property characteristic; date has been replaced by date features.
model_df = model_df.drop(columns=['id', 'date'])
model_df['zipcode'] = model_df['zipcode'].astype(str)

X = model_df.drop(columns='price')
y = model_df['price']
categorical_features = ['zipcode']
numeric_features = [col for col in X.columns if col not in categorical_features]

preprocessor = ColumnTransformer([
    ('numeric', Pipeline([('imputer', SimpleImputer(strategy='median'))]), numeric_features),
    ('categorical', Pipeline([
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('encoder', OneHotEncoder(handle_unknown='ignore'))
    ]), categorical_features)
])

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42)
print('Training set:', X_train.shape, 'Testing set:', X_test.shape)

## 6. Train and evaluate three regression models

In [ ]:
models = {
    'Linear Regression': LinearRegression(),
    'Decision Tree Regressor': DecisionTreeRegressor(max_depth=20, min_samples_leaf=3, random_state=42),
    'Random Forest Regressor': RandomForestRegressor(n_estimators=150, min_samples_leaf=2, random_state=42, n_jobs=-1)
}

results = []
fitted_models = {}
for name, model in models.items():
    pipeline = Pipeline([('preprocessor', preprocessor), ('model', model)])
    pipeline.fit(X_train, y_train)
    predictions = pipeline.predict(X_test)
    mse = mean_squared_error(y_test, predictions)
    results.append({
        'Model': name,
        'MAE': mean_absolute_error(y_test, predictions),
        'MSE': mse,
        'RMSE': np.sqrt(mse),
        'R² Score': r2_score(y_test, predictions)
    })
    fitted_models[name] = pipeline

results_df = pd.DataFrame(results).sort_values('R² Score', ascending=False).reset_index(drop=True)
display(results_df.style.format({'MAE': '${:,.0f}', 'MSE': '{:,.0f}', 'RMSE': '${:,.0f}', 'R² Score': '{:.4f}'}))

## 7. Model comparison and conclusion

In [ ]:
plt.figure(figsize=(10, 5))
sns.barplot(data=results_df, x='R² Score', y='Model', palette='viridis')
plt.title('Model Comparison: R² Score')
plt.xlim(0, 1)
plt.show()

best_model = results_df.iloc[0]
print(f"Best model: {best_model['Model']}")
print(f"It achieved the highest R² score ({best_model['R² Score']:.4f}) and an RMSE of ${best_model['RMSE']:,.0f}.")
print('This model is selected because it explains the largest share of variation in house prices on unseen test data.')